## Dataset Embedding Experiments with clustering and cross encoder

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
import pandas as pd

In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder

bi_encoder = SentenceTransformer('all-mpnet-base-v2')  

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')



In [4]:
jd_df = pd.read_excel("../1_data_cleaning/filtered_jd_sections2.xlsx")


In [5]:
jd_df.head()

,job_description,location_cleaned,job_title,jd_duties,jd_requirements,jd_education
0,Job SummaryDo you have a strong aptitude for w...,"Natick, MA",Content Developer for MATLAB Code Generation,Software components make up an ever larger par...,Job SummaryDo you have a strong aptitude for w...,Minimum Qualifications A bachelor's degree and...
1,Overview External: Chevron is one of the world...,"Houston, TX",Land Assistant,Overview External: Chevron is one of the world...,"Prepare, and secure appropriate approvals for ...",Preferred Qualifications: Bachelor's degree
2,Overview: The Hartman Non-Profit is seeking to...,"Houston, TX",Project Manager,Overview: The Hartman Non-Profit is seeking to...,Required Abilities and Experience: · A strong ...,· A minimum of a Bachelor’s degree in a busine...
3,City: Houston State:Texas Postal/Zip Code: 770...,"Houston, TX",Asphalt Quality Control Manager -Houston,Our operations span the nation from Montana to...,Other Requirements Display a professional and ...,Qualifications Bachelor’s degree (B
4,Hiring! Immediate openings for Customer Suppor...,"San Antonio, TX",Customer Support Specialist!,Immediate openings for Customer Support Specia...,Hiring\nWe want your excellent customer servic...,High School Diploma/GED


In [6]:
# create semantic sentence for each row
def jd_build_semantic_sentence(row):
    desc = str(row.get("job_description", ""))
    duties = str(row.get("jd_duties", ""))
    req = str(row.get("jd_requirements", ""))
    edu = str(row.get("jd_education", ""))
    title = str(row.get("job_title", ""))

    return (
        f"Job Posting:\n"
        f"- Job Title: {title}\n"
        f"- Description: {desc}\n"
        f"- Responsibilities: {duties}\n"
        f"- Requirements: {req}\n"
        f"- Preferred Education: {edu}"
    )



jd_df["semantic_sentence"] = jd_df.apply(jd_build_semantic_sentence, axis=1)
jd_df.head()



,job_description,location_cleaned,job_title,jd_duties,jd_requirements,jd_education,semantic_sentence
0,Job SummaryDo you have a strong aptitude for w...,"Natick, MA",Content Developer for MATLAB Code Generation,Software components make up an ever larger par...,Job SummaryDo you have a strong aptitude for w...,Minimum Qualifications A bachelor's degree and...,Job Posting:\n- Job Title: Content Developer f...
1,Overview External: Chevron is one of the world...,"Houston, TX",Land Assistant,Overview External: Chevron is one of the world...,"Prepare, and secure appropriate approvals for ...",Preferred Qualifications: Bachelor's degree,Job Posting:\n- Job Title: Land Assistant\n- D...
2,Overview: The Hartman Non-Profit is seeking to...,"Houston, TX",Project Manager,Overview: The Hartman Non-Profit is seeking to...,Required Abilities and Experience: · A strong ...,· A minimum of a Bachelor’s degree in a busine...,Job Posting:\n- Job Title: Project Manager\n- ...
3,City: Houston State:Texas Postal/Zip Code: 770...,"Houston, TX",Asphalt Quality Control Manager -Houston,Our operations span the nation from Montana to...,Other Requirements Display a professional and ...,Qualifications Bachelor’s degree (B,Job Posting:\n- Job Title: Asphalt Quality Con...
4,Hiring! Immediate openings for Customer Suppor...,"San Antonio, TX",Customer Support Specialist!,Immediate openings for Customer Support Specia...,Hiring\nWe want your excellent customer servic...,High School Diploma/GED,Job Posting:\n- Job Title: Customer Support Sp...


In [7]:
resume_df = pd.read_csv("../1_data_cleaning/resume_cleaned.csv")

In [8]:
resume_df.head(10)

,Uniq Id,Work Experience,Education,Skills,Additional Information
0,3ddb29e616f31947053b257f327969d7,"Sales Manager-MadgeTech, Inc-August 2015 to Fe...",B.A.-History-Franklin Pierce University-Rindge...,"120 months-CRM,72 months-Contract Negotiation,...",• Well-Developed Sales & Business Acumen ...
1,9138476c76bcbbefadedd4862966c3d2,Implementation Engineer-Versatile Communicatio...,"--ShoreTel University-Austin, TX|Master-PC & N...","15 months-CISCO,12 months-FIBER OPTIC,6 months...","TECHNICAL SKILLS\n\nHardware: Switches, Router..."
2,cd1cafa706f917a627982bf47291b888,Education Information Dissemination Coordinato...,"Bachelor's-Management-Regis College-Weston, MA",NaN,NaN
3,53aea69598c6c1084e4bce89f0494bc3,Engineering Department Intern-Town of Billeric...,Bachelor of Science-Civil and Environmental En...,NaN,• Bachelors of Science in Civil and Environmen...
4,90f8f99d66ebc6c09fceee37aff14bc1,Pack and Ship/SORT Technician-Intel Corporatio...,BS-Information Technology-University of Massac...,NaN,Engineering Technician/Planning Analyst/Operat...
5,fb1c760fd5311dad01e950f3062a9296,BDC Data Analyst-Gary Rome Auto Group-January ...,Bachelor of Science-Psychology-University of P...,"30 months-SIX-SIGMA,36 months-DATA ANALYSIS,24...",CORE COMPETENCIES\n• Project Management\t\t• T...
6,7795cdc183ab88aaffec73a5d99a1b82,Safety Engineer Intern-Hexagon Manufacturing I...,Master of Science-Systems Engineering-UNIVERSI...,"9 months-MATLAB,36 months-OPTIMIZATION,36 mont...","Core Competencies: Control Systems, Automotive..."
7,e219a144eca8463f5d70fdec9426466f,Classified Ads Manager-Quality of Life Publica...,NaN,NaN,NaN
8,c7c409fe9652e1eae9ecb05cce05426e,ASSISTANT PROGRAM MANAGER-HARBOR HOMES-August ...,CPR certified--SOUTHERN NEW HAMPSHIRE UNIVERISTY-,"13 months-PROGRAM MANAGER,0 months-RETAIL,13 m...",Skills & Abilities\nMANAGEMENT\n• 4 years of m...
9,7f562eaaa992790efe423a197bfd0ffa,Technical Customer Service-SmartCo Services LL...,Master of Science-EILCO-Engineering school of ...,NaN,NaN


In [9]:
# create semantic sentences for resumes
def resume_build_semantic_sentence(row):
    work_exp = str(row.get("Work Experience", ""))
    skills = str(row.get("Skills", ""))
    edu = str(row.get("Education", ""))
    add_info = str(row.get("Additional Information", ""))

    return (
        f"Candidate Profile:\n"
        f"- Experience: {work_exp}\n"
        f"- Skills: {skills}\n"
        f"- Education: {edu}\n"
        f"- Additional Info: {add_info}"
    )



In [10]:
# create semantic sentences for resumes
resume_df["semantic_sentence"] = resume_df.apply(resume_build_semantic_sentence, axis=1)
resume_df.head()


,Uniq Id,Work Experience,Education,Skills,Additional Information,semantic_sentence
0,3ddb29e616f31947053b257f327969d7,"Sales Manager-MadgeTech, Inc-August 2015 to Fe...",B.A.-History-Franklin Pierce University-Rindge...,"120 months-CRM,72 months-Contract Negotiation,...",• Well-Developed Sales & Business Acumen ...,Candidate Profile:\n- Experience: Sales Manage...
1,9138476c76bcbbefadedd4862966c3d2,Implementation Engineer-Versatile Communicatio...,"--ShoreTel University-Austin, TX|Master-PC & N...","15 months-CISCO,12 months-FIBER OPTIC,6 months...","TECHNICAL SKILLS\n\nHardware: Switches, Router...",Candidate Profile:\n- Experience: Implementati...
2,cd1cafa706f917a627982bf47291b888,Education Information Dissemination Coordinato...,"Bachelor's-Management-Regis College-Weston, MA",NaN,NaN,Candidate Profile:\n- Experience: Education In...
3,53aea69598c6c1084e4bce89f0494bc3,Engineering Department Intern-Town of Billeric...,Bachelor of Science-Civil and Environmental En...,NaN,• Bachelors of Science in Civil and Environmen...,Candidate Profile:\n- Experience: Engineering ...
4,90f8f99d66ebc6c09fceee37aff14bc1,Pack and Ship/SORT Technician-Intel Corporatio...,BS-Information Technology-University of Massac...,NaN,Engineering Technician/Planning Analyst/Operat...,Candidate Profile:\n- Experience: Pack and Shi...


In [11]:
# add index columns for both datasets
jd_df["jd_index"] = jd_df.index
resume_df["resume_index"] = resume_df.index


In [12]:
def embed_sentence(sentence):
    return bi_encoder.encode(sentence)


In [13]:
# embed JD semantic sentences
jd_df["semantic_emb"] = jd_df["semantic_sentence"].apply(embed_sentence)
jd_df.head()

,job_description,location_cleaned,job_title,jd_duties,jd_requirements,jd_education,semantic_sentence,jd_index,semantic_emb
0,Job SummaryDo you have a strong aptitude for w...,"Natick, MA",Content Developer for MATLAB Code Generation,Software components make up an ever larger par...,Job SummaryDo you have a strong aptitude for w...,Minimum Qualifications A bachelor's degree and...,Job Posting:\n- Job Title: Content Developer f...,0,"[-0.010107441, -0.023181697, -0.041947138, -0...."
1,Overview External: Chevron is one of the world...,"Houston, TX",Land Assistant,Overview External: Chevron is one of the world...,"Prepare, and secure appropriate approvals for ...",Preferred Qualifications: Bachelor's degree,Job Posting:\n- Job Title: Land Assistant\n- D...,1,"[0.040389102, 0.07183061, -0.0014597354, -0.04..."
2,Overview: The Hartman Non-Profit is seeking to...,"Houston, TX",Project Manager,Overview: The Hartman Non-Profit is seeking to...,Required Abilities and Experience: · A strong ...,· A minimum of a Bachelor’s degree in a busine...,Job Posting:\n- Job Title: Project Manager\n- ...,2,"[0.036204915, 0.08343988, -0.005588149, 0.0499..."
3,City: Houston State:Texas Postal/Zip Code: 770...,"Houston, TX",Asphalt Quality Control Manager -Houston,Our operations span the nation from Montana to...,Other Requirements Display a professional and ...,Qualifications Bachelor’s degree (B,Job Posting:\n- Job Title: Asphalt Quality Con...,3,"[-0.015358253, 0.04020662, -0.009223265, 0.023..."
4,Hiring! Immediate openings for Customer Suppor...,"San Antonio, TX",Customer Support Specialist!,Immediate openings for Customer Support Specia...,Hiring\nWe want your excellent customer servic...,High School Diploma/GED,Job Posting:\n- Job Title: Customer Support Sp...,4,"[0.018529275, -0.050914153, -0.04652271, -0.04..."


In [14]:
# embed resume semantic sentences
resume_df["semantic_emb"] = resume_df["semantic_sentence"].apply(embed_sentence)
resume_df.head()

,Uniq Id,Work Experience,Education,Skills,Additional Information,semantic_sentence,resume_index,semantic_emb
0,3ddb29e616f31947053b257f327969d7,"Sales Manager-MadgeTech, Inc-August 2015 to Fe...",B.A.-History-Franklin Pierce University-Rindge...,"120 months-CRM,72 months-Contract Negotiation,...",• Well-Developed Sales & Business Acumen ...,Candidate Profile:\n- Experience: Sales Manage...,0,"[0.059411276, 0.053221624, -0.01228166, -0.066..."
1,9138476c76bcbbefadedd4862966c3d2,Implementation Engineer-Versatile Communicatio...,"--ShoreTel University-Austin, TX|Master-PC & N...","15 months-CISCO,12 months-FIBER OPTIC,6 months...","TECHNICAL SKILLS\n\nHardware: Switches, Router...",Candidate Profile:\n- Experience: Implementati...,1,"[0.03360335, 0.04754441, -0.024735957, -0.0731..."
2,cd1cafa706f917a627982bf47291b888,Education Information Dissemination Coordinato...,"Bachelor's-Management-Regis College-Weston, MA",NaN,NaN,Candidate Profile:\n- Experience: Education In...,2,"[0.06029162, 0.045489, 0.0030238933, -0.039945..."
3,53aea69598c6c1084e4bce89f0494bc3,Engineering Department Intern-Town of Billeric...,Bachelor of Science-Civil and Environmental En...,NaN,• Bachelors of Science in Civil and Environmen...,Candidate Profile:\n- Experience: Engineering ...,3,"[-0.0609074, 0.069869526, -0.015645016, 0.0020..."
4,90f8f99d66ebc6c09fceee37aff14bc1,Pack and Ship/SORT Technician-Intel Corporatio...,BS-Information Technology-University of Massac...,NaN,Engineering Technician/Planning Analyst/Operat...,Candidate Profile:\n- Experience: Pack and Shi...,4,"[0.023709496, 0.005459349, -0.03795984, -0.048..."


In [15]:
# clustering embeddings for both datasets
from sklearn.cluster import KMeans
import numpy as np

all_embs = np.vstack([
    np.vstack(resume_df["semantic_emb"].to_list()),
    np.vstack(jd_df["semantic_emb"].to_list())
])


NUM_CLUSTERS = 10

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
cluster_labels = kmeans.fit_predict(all_embs)


resume_clusters = cluster_labels[:len(resume_df)]
jd_clusters = cluster_labels[len(resume_df):]

resume_df["cluster"] = resume_clusters
jd_df["cluster"] = jd_clusters


In [16]:
# convert to numpy array
import numpy as np

resume_df["semantic_emb"] = resume_df["semantic_emb"].apply(lambda x: np.array(x))
jd_df["semantic_emb"] = jd_df["semantic_emb"].apply(lambda x: np.array(x))


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

def get_coarse_candidates(resume_embedding, resume_cluster, jd_df, top_k=50):
    candidates = jd_df[jd_df["cluster"] == resume_cluster].copy()

    sims = cosine_similarity([resume_embedding], list(candidates["semantic_emb"]))[0]
    candidates["sim"] = sims


    candidates = candidates.sort_values("sim", ascending=False).head(top_k)

    return candidates


In [18]:
def rerank_with_cross_encoder(resume_sentence, candidate_df, cross_encoder, top_k=3):
    if candidate_df is None or len(candidate_df) == 0:
        return []

    pairs = [
        (resume_sentence, jd_sent)
        for jd_sent in candidate_df["semantic_sentence"]
    ]

    # cross-encoder predict
    scores = cross_encoder.predict(pairs)

    candidate_df = candidate_df.copy()
    candidate_df["ce_score"] = scores

    top_matches = candidate_df.sort_values("ce_score", ascending=False).head(top_k)

    return top_matches[["jd_index", "job_title", "location_cleaned", "ce_score"]]


In [19]:
def match_resume(resume_row, jd_df, k_coarse=50, k_final=3):
    resume_emb = np.array(resume_row["semantic_emb"])
    resume_cluster = resume_row["cluster"]
    resume_sentence = resume_row["semantic_sentence"]

    # Step 1: coarse retrieve
    candidates = get_coarse_candidates(
        resume_embedding=resume_emb,
        resume_cluster=resume_cluster,
        jd_df=jd_df,
        top_k=k_coarse
    )

    # Step 2: cross encoder rerank
    final_matches = rerank_with_cross_encoder(
        resume_sentence=resume_sentence,
        candidate_df=candidates,
        cross_encoder=cross_encoder,
        top_k=k_final
    )

    return final_matches


In [21]:
for i in range(5):
    row = resume_df.iloc[i]
    top_matches = match_resume(row, jd_df)

    print(f"================ Resume #{i} ================")
    print(row["Work Experience"][:400], "...\n")
    print(top_matches)
    print("-----------------------------------------------------\n")


================ Resume #0 ================
Sales Manager-MadgeTech, Inc-August 2015 to February 2017-Warner, NH-•       Built and Directed inside sales team offering solution-based data logging hardware, software, and services to customers across multiple vertical markets

•       Directed personnel in support of global channel partner network

•       15% Sales Growth from 2015 to 2017: $8.5M to $9.7M (Combined channel and direct)

•      ...

      jd_index                        job_title location_cleaned  ce_score
871        871           Director of Purchasing       Newark, NJ  1.678213
1005      1005  Regional Sales Manager - Dallas       Dallas, TX  1.497836
602        602                    Store Manager  San Antonio, TX  1.313631
-----------------------------------------------------

================ Resume #1 ================
Implementation Engineer-Versatile Communications-January 2016 to September 2017-Marlborough, MA-• Customer Facing implementation specialist for deployi